In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import nltk
import pickle

In [2]:
df = pd.read_csv("spam.csv", encoding='latin1',sep="\t",on_bad_lines="skip",header=None)
df.head()


,0,1
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
df = df.iloc[:, :2]
df.columns = ['label', 'message']
df.head()


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['label'])
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
df.drop_duplicates(inplace=True)
df.shape

(5169, 2)

In [6]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')



[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\91892\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91892\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

In [8]:
def transform_text(text):

    text = text.lower()

    words = nltk.word_tokenize(text)

    y = []

    for i in words:
        if i.isalnum():
            y.append(i)

    words = y[:]
    y.clear()

    for i in words:
        if i not in stopwords.words('english'):
            y.append(i)

    words = y[:]
    y.clear()

    for i in words:
        y.append(ps.stem(i))

    return " ".join(y)

In [9]:
transform_text("WIN FREE MONEY NOW!!!")

'win free money'

In [10]:
df['transformed_text'] = df['message'].apply(transform_text)
df.head()

,label,message,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [11]:
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['label']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
model = MultinomialNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [14]:
accuracy_score(y_test, y_pred)

0.9748549323017408

In [15]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[894   0]
 [ 26 114]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       894
           1       1.00      0.81      0.90       140

    accuracy                           0.97      1034
   macro avg       0.99      0.91      0.94      1034
weighted avg       0.98      0.97      0.97      1034



In [16]:
import pickle


In [17]:
pickle.dump(
    tfidf,
    open("vectorizer.pkl","wb")
)
pickle.dump(
    model,
    open("model.pkl","wb")
)

In [18]:
import nltk
nltk.download('puntk')
nltk.download('stopwords')

[nltk_data] Error loading puntk: Package 'puntk' not found in index
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91892\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True